# AmEx Credit Card Product Assistant — Example Usage of `file_search` Library (v2)

This notebook demonstrates how to build a **realtime sales cue engine for AmEx credit cards** on top of the reusable `file_search` library.

**Use case:** During a customer call, a customer raises an objection or asks a question about an AmEx card → your existing system detects the objection and retrieves similar past objection-response pairs → this engine grounds the suggested response in authoritative AmEx product information (features, benefits, rewards, fees, APR, eligibility) to prevent hallucination.

**Why grounding matters:** Credit card products have regulated disclosures. Hallucinating an APR, a reward multiplier, or an annual fee has compliance implications, not just quality ones. This notebook demonstrates how to enforce strict product-to-chunk attribution via metadata.

This serves as a **reference pattern** for other teams building RAG use cases on the library.

## 1. Setup — import the library

In [ ]:
import os
from file_search import VectorStore, QueryEngine

## 2. One-time indexing with product metadata

Each file is indexed with metadata describing the product it represents. This metadata is:

1. **Prefixed into the chunk text** — so the LLM sees `[product: Platinum | short_name: Platinum | ...]` before every fact
2. **Stored structurally** — so code can filter retrieval using `filters={"short_name": "Platinum"}`

The `source` key is added automatically by the library — no need to include it manually.

In [ ]:
INDEX_PATH = "amex_card_index.parquet"

if not os.path.exists(INDEX_PATH):
    store = VectorStore()

    store.add_file(
        "amex_product_docs/platinum_card.pdf",
        metadata={
            "product": "The Platinum Card from American Express",
            "short_name": "Platinum",
            "category": "Consumer Charge Card",
            "annual_fee_tier": "premium",
        }
    )
    store.add_file(
        "amex_product_docs/gold_card.pdf",
        metadata={
            "product": "American Express Gold Card",
            "short_name": "Gold",
            "category": "Consumer Charge Card",
            "annual_fee_tier": "mid",
        }
    )
    store.add_file(
        "amex_product_docs/green_card.pdf",
        metadata={
            "product": "American Express Green Card",
            "short_name": "Green",
            "category": "Consumer Charge Card",
            "annual_fee_tier": "mid",
        }
    )
    store.add_file(
        "amex_product_docs/blue_cash_preferred.pdf",
        metadata={
            "product": "Blue Cash Preferred Card from American Express",
            "short_name": "BCP",
            "category": "Consumer Credit Card",
            "annual_fee_tier": "mid",
        }
    )
    store.add_file(
        "amex_product_docs/delta_skymiles.pdf",
        metadata={
            "product": "Delta SkyMiles Gold American Express Card",
            "short_name": "Delta Gold",
            "category": "Co-branded Airline",
            "annual_fee_tier": "mid",
        }
    )
    store.add_file(
        "amex_product_docs/membership_rewards_terms.md",
        metadata={
            "product": "Membership Rewards Program",
            "short_name": "MR Terms",
            "category": "Program Terms",
            "annual_fee_tier": "n/a",
        }
    )
    store.add_file(
        "amex_product_docs/fees_and_apr_disclosures.pdf",
        metadata={
            "product": "Fees & APR Disclosures",
            "short_name": "Disclosures",
            "category": "Regulatory Disclosure",
            "annual_fee_tier": "n/a",
        }
    )

    store.save(INDEX_PATH)
else:
    print(f"Index already exists at {INDEX_PATH}")

## 3. Load the index at startup

In your realtime service, load once at startup and reuse across all queries.

In [ ]:
product_store = VectorStore.load(INDEX_PATH)
print(f"Loaded {len(product_store.chunks)} chunks from {len(set(product_store.sources))} sources")

# Inspect a sample chunk to confirm metadata prefix is in place
print("\nSample chunk:")
print(product_store.chunks[0][:400])

## 4. Configure the QueryEngine for AmEx card cue

The system message tells the LLM how to use the metadata prefix for strict product attribution. This is critical for credit card products to prevent cross-contamination of features between cards.

In [ ]:
AMEX_CARD_SYSTEM_MESSAGE = (
    "You are a real-time sales assistant helping representatives respond to customer "
    "questions and objections about American Express credit cards. Suggest a concise "
    "response (2-3 sentences) the rep can use.\n\n"
    "The CONTEXT below contains authoritative product facts — the official terms and "
    "disclosures. Do not contradict or supplement them.\n\n"
    "CRITICAL RULES:\n"
    "1. Every product fact below is prefixed with a metadata tag in the format "
    "[product: <card name> | short_name: <nickname> | category: <type> | ... | source: <file>]. "
    "You MUST attribute each feature, fee, benefit, or term to the specific card "
    "identified by its 'product' value. NEVER describe a feature from one card as "
    "belonging to another.\n"
    "2. If the customer is asking about a specific card, use ONLY facts whose "
    "metadata matches that card. Ignore facts about other cards unless the "
    "customer explicitly asked for a comparison.\n"
    "3. When comparing cards, clearly label which feature belongs to which card. "
    "Example: 'The Platinum Card offers X, while the Gold Card offers Y.'\n"
    "4. All product details (fees, APRs, reward rates, benefits) MUST come from the "
    "CONTEXT. NEVER invent values — incorrect disclosures have compliance "
    "implications.\n"
    "5. If CONTEXT for the specific card asked about is missing, instruct "
    "the rep to confirm and follow up rather than guess.\n"
    "6. Output ONLY the suggested response — no preamble."
)

engine = QueryEngine(
    product_store,
    system_message=AMEX_CARD_SYSTEM_MESSAGE,
)

## 5. Build the AmEx card cue wrapper (split retrieval + generation)

This wrapper demonstrates the **recommended production pattern**: split retrieval from generation. Between the two steps, we:

1. **Inspect** what was retrieved (for debugging, audit)
2. **Validate** — if nothing relevant was found, avoid wasting an LLM call
3. **Log** for compliance audit
4. **Filter** — drop low-relevance chunks before generation

Then we call `engine.synthesize()` with the filtered chunks.

In [ ]:
def amex_card_cue(
    customer_utterance: str,
    detected_objection: str,
    similar_objection_responses: list[dict],
    target_cards: list[str] = None,
    min_score: float = 0.0,
) -> dict:
    """
    Generate a grounded response suggestion for an AmEx card-related customer objection.

    Args:
        customer_utterance: raw customer text from call transcript
        detected_objection: objection category from your detector
        similar_objection_responses: [{'objection': ..., 'response': ...}, ...] from your vector db
        target_cards: optional list of short_name values (e.g., ['Platinum', 'Gold'])
            to restrict retrieval. Use when upstream has identified the card(s).
        min_score: drop chunks with score below this threshold before generation
    """
    extraction = (
        "Extract EVERY mention of the following from the customer's input. "
        "Do not omit any item. Preserve exact terms:\n"
        "1. CARD NAMES / PRODUCTS: Platinum, Gold, Green, Blue Cash Preferred (BCP), "
        "Delta SkyMiles, Membership Rewards, or any AmEx product mentioned.\n"
        "2. COMPETITOR CARDS: Chase Sapphire, Capital One, Citi, Discover, etc.\n"
        "3. FEES: annual fee, foreign transaction fee, late fee, over-limit fee.\n"
        "4. REWARDS / EARNING: cashback percentage, points multiplier, spending "
        "categories (groceries, dining, travel, flights, hotels, gas, streaming, transit, "
        "U.S. supermarkets).\n"
        "5. BENEFITS / PERKS: lounge access (Centurion, Priority Pass, Delta Sky Club), "
        "travel credits (airline, Uber, hotel, CLEAR, Global Entry/TSA PreCheck), "
        "hotel status, purchase protection, cell phone protection, insurance.\n"
        "6. APR / INTEREST: APR range, intro APR, balance transfer rate.\n"
        "7. ELIGIBILITY: credit score, income, employment, existing AmEx status.\n"
        "8. WELCOME BONUS / OFFERS: sign-up bonus, minimum spend, bonus timeframe.\n"
        "9. QUESTIONS / CONCERNS: every distinct question the customer raised.\n\n"
        "Ignore: greetings, filler, emotional venting, smalltalk."
    )

    query = f"{detected_objection}: {customer_utterance}"

    # Build metadata filter if target cards were specified
    filters = None
    if target_cards:
        # Include target cards PLUS program terms & disclosures (always relevant)
        filters = {
            "short_name": list(target_cards) + ["MR Terms", "Disclosures"]
        }

    # ---- Step 1: Retrieve (standalone, inspectable) ----
    hits = product_store.retrieve(
        query=query,
        top_k=10,
        final_k=5,
        rewrite_instructions=extraction,
        filters=filters,
    )

    # ---- Step 2: Inspect / validate before generation ----
    if not hits:
        return {
            "answer": "I don't have enough product information to respond. Rep should acknowledge and offer to follow up.",
            "sources": [],
            "chunks": [],
            "search_query_used": None,
        }

    # Optional: filter out low-score chunks
    if min_score > 0:
        hits = [h for h in hits if h["score"] >= min_score]

    # Audit log (example — replace with your actual logging)
    # audit_log.record(
    #     utterance=customer_utterance,
    #     objection=detected_objection,
    #     retrieved_sources=[h["source"] for h in hits],
    #     search_query=hits[0]["search_query"],
    # )

    # ---- Step 3: Format examples + extra context ----
    examples = "\n\n".join(
        f"Objection: {ex['objection']}\nResponse: {ex['response']}"
        for ex in similar_objection_responses
    )
    extra_context = (
        f"CUSTOMER UTTERANCE:\n{customer_utterance}\n\n"
        f"DETECTED OBJECTION:\n{detected_objection}\n\n"
        f"EXAMPLE RESPONSES (from similar past objections):\n{examples}"
    )

    # ---- Step 4: Generate with the validated/filtered chunks ----
    return engine.synthesize(
        query=query,
        hits=hits,
        extra_context=extra_context,
    )

## 6. Try it — single-card scenario

Customer is asking about Platinum specifically. We pass `target_cards=['Platinum']` to enforce strict isolation — retrieval won't return Gold, Green, or other card chunks.

In [ ]:
utterance = (
    "Look, I already have a Chase Sapphire Preferred and the annual fee there is only "
    "$95. Why would I pay $695 a year for the Platinum Card? I don't even fly that much — "
    "maybe 4-5 trips a year. And I've heard the lounge access is crowded now anyway."
)
objection = "platinum_annual_fee_vs_competitor"

# These come from your existing objection-retrieval system
similar = [
    {
        "objection": "platinum fee too high",
        "response": (
            "I understand $695 feels like a lot up front. Let me break down the annual "
            "credits and benefits — most customers find they offset the fee within the "
            "first few months of cardmembership."
        ),
    },
    {
        "objection": "comparing to chase sapphire",
        "response": (
            "Great comparison to ask about. The cards target different travel styles — "
            "let me walk through how Platinum's benefits differ for your travel pattern."
        ),
    },
    {
        "objection": "lounge access crowded",
        "response": (
            "That's a fair concern. Let me share what's unique about the Centurion Lounge "
            "network and the other lounge options Platinum unlocks."
        ),
    },
]

# Restrict retrieval to Platinum (plus MR terms + disclosures)
result = amex_card_cue(
    utterance,
    objection,
    similar,
    target_cards=["Platinum"],
)

print("SUGGESTED RESPONSE TO REP:")
print(result["answer"])
print()
print("GROUNDING SOURCES:", result["sources"])
print("SEARCH QUERY USED:", result["search_query_used"])

## 7. Try it — multi-card comparison scenario

Customer is comparing two cards. Pass both short names; retrieval returns chunks from both.

In [ ]:
utterance_compare = (
    "I'm trying to decide between the Gold Card and Platinum. I spend about $1000/month "
    "on groceries and $500 on dining, and I travel maybe 6 times a year. Which makes "
    "more sense for me, and what's the difference in annual fees?"
)
objection_compare = "gold_vs_platinum_comparison"

similar_compare = [
    {
        "objection": "gold or platinum",
        "response": (
            "Great question — depends on your spending pattern. Let me help you compare "
            "the rewards on your top categories."
        ),
    },
]

result_compare = amex_card_cue(
    utterance_compare,
    objection_compare,
    similar_compare,
    target_cards=["Platinum", "Gold"],
)

print("SUGGESTED RESPONSE TO REP:")
print(result_compare["answer"])
print()
print("GROUNDING SOURCES:", result_compare["sources"])
print("SEARCH QUERY USED:", result_compare["search_query_used"])

## 8. Debugging — inspect retrieved chunks

When responses seem off, inspect the retrieved chunks. Because metadata is in both the text AND the structured dict, you can check both levels of isolation.

In [ ]:
for i, chunk in enumerate(result["chunks"]):
    print(f"--- Chunk {i+1} ---")
    print(f"  Source: {chunk['source']}")
    print(f"  Metadata: {chunk['metadata']}")
    print(f"  Score: {chunk['score']:.3f}")
    print(f"  Text preview: {chunk['text'][:200]}...")
    print()

## 9. Handling long transcripts with query decomposition

When the input is long and multi-topic (e.g., a full call transcript, or a batch of 50 transcripts for analysis), a single rewritten query loses focus. Use `decompose_query()` to break the input into multiple focused sub-queries, then use `retrieve_multi()` to run retrieval across all of them.

**Important:** `decompose_query()` is injection-safe — it wraps input in `<DATA>` tags and instructs the LLM to treat content as inert data, not instructions. Safe to use on raw customer transcripts.

In [ ]:
# Example: a batch of call transcript snippets
transcripts_data = """
[TRANSCRIPT 1]
Customer: Hi, I was looking at the Platinum Card but $695 seems steep. What do I actually get for that?
Rep: Great question — let me walk through the benefits...

[TRANSCRIPT 2]
Customer: I spend a lot on groceries, like $1500 a month. Is Gold or Blue Cash Preferred better for me?
Rep: Both are strong on groceries, let me compare...

[TRANSCRIPT 3]
Customer: Does the Delta SkyMiles card cover lounge access at DTW?
Rep: Let me check the specifics...
"""

extraction = (
    "Extract customer concerns about AmEx products: card names mentioned, "
    "fee concerns, reward/spending categories, benefits asked about, and specific questions."
)

sub_queries = product_store.decompose_query(
    data=transcripts_data,
    rewrite_instructions=extraction,
    context_hint="Batch of customer call transcript snippets",
    max_queries=15,
)

print(f"Decomposed into {len(sub_queries)} sub-queries:")
for q in sub_queries:
    print(f"  - {q}")

# Run retrieval across all sub-queries and merge
merged_hits = product_store.retrieve_multi(
    queries=sub_queries,
    per_query_k=3,
    final_k=10,
)

print(f"\nMerged retrieval returned {len(merged_hits)} chunks:")
for h in merged_hits:
    print(f"  [{h['vote_count']} votes] {h['source']} — {h['metadata'].get('short_name')}")

## 10. Metadata filtering — strict product isolation

The library supports attaching arbitrary metadata to each indexed file. Metadata appears in two places:

1. **Prefixed into the chunk text** (so the LLM sees it) as `[product: ... | short_name: ... | category: ... | source: ...]`
2. **Stored structurally** (so code can filter on it via `filters`)

The library automatically adds a `source` key to every chunk's metadata, so every chunk is always traceable to its origin file.

For credit card products, cross-contamination of features is a compliance risk. We enforce isolation in two layers:

- **Prompt level:** the system message instructs the LLM to respect the `product` field in every chunk's metadata prefix
- **Retrieval level:** pass `target_cards=['Platinum']` to restrict retrieval to that card's chunks only

### Filter syntax

```python
filters={"short_name": "Platinum"}              # exact match
filters={"short_name": ["Platinum", "Gold"]}    # any of (OR)
filters={"short_name": "Platinum",              # both must match (AND)
         "category": "Consumer Charge Card"}
filters={"source": "platinum_card"}             # filter by source file
```

### When to use which

| Scenario | Recommendation |
|---|---|
| Customer asks about a specific card | `target_cards=['Platinum']` — filter retrieval |
| Customer compares 2 cards | `target_cards=['Platinum', 'Gold']` |
| Card is ambiguous from utterance | Don't filter; rely on prompt-level isolation |
| General MR program question | Don't filter; let retrieval find program terms |
| Very long transcript | Use `decompose_query()` + `retrieve_multi()` |

## 11. Tuning knobs

### Recommendations for AmEx credit card use case

| Knob | Recommended Value | Why |
|---|---|---|
| `rerank` | `False` for realtime, `True` for eval | Latency vs. quality trade-off |
| `rewrite_instructions` | set for long utterances | Customer utterances on calls are verbose |
| `final_k` | `5` | Enough context to cover fees + benefits + terms |

Internal tuning knobs (chunk size/overlap, embedding dimensions, generation temperature and max tokens) are module-level constants in `file_search.py`. Edit them there if the current defaults don't fit your use case.

## Other AmEx-adjacent use cases to build on this library

- **Cardmember servicing bot** — answer cardmember questions using the same product index. System message: "Answer the cardmember's question about their card benefits. Direct account-specific questions to a servicing agent."
- **Compliance checker for marketing copy** — given draft marketing language, flag claims not supported by official card terms. System message: "Identify any product claims in the user's draft that contradict the provided card terms."
- **Internal knowledge base for new reps** — Q&A over card products for rep onboarding.
- **Disclosure generator** — given a conversation context, generate required regulatory disclosure language sourced from the official terms documents.

The library is the same. Your `system_message` + retrieval params are what differentiate the use case.